In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv("../data/processed/heart_clean.csv")

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numerical_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix

solver_list = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky']
best_solver = None
best_score = 0
print("Comparaison des solveurs  :")
for s in solver_list:
    lr = LogisticRegression(solver=s, max_iter=1000, random_state=42)
    scores = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='recall')
    mean_score = scores.mean()
    print(f"  Solver {s:15s} → Rappel moyen : {mean_score:.4f}")
    
    if mean_score > best_score:
        best_score = mean_score
        best_solver = s

print(f"\n Meilleur solveur : {best_solver} ")

lr_final = LogisticRegression(solver=best_solver, max_iter=1000, class_weight='balanced', random_state=42)
lr_final.fit(X_train_scaled, y_train)

y_pred = lr_final.predict(X_test_scaled)

print(f"Rappel (Recall) : {recall_score(y_test, y_pred):.4f}")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred))

Comparaison des solveurs  :
  Solver lbfgs           → Rappel moyen : 0.8718
  Solver liblinear       → Rappel moyen : 0.8718
  Solver newton-cg       → Rappel moyen : 0.8718
  Solver newton-cholesky → Rappel moyen : 0.8718

 Meilleur solveur : lbfgs 
Rappel (Recall) : 0.8333
Accuracy : 0.8424
Matrice de confusion :
 [[70 12]
 [17 85]]


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix

dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)

param_grid = {
    'max_depth': [3, 4, 5, 6, 7, 8],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [1, 2, 3, 4]
}

grid_dt = GridSearchCV(dt, param_grid, cv=5, scoring='recall', n_jobs=-1)
grid_dt.fit(X_train_scaled, y_train)

print(f" Meilleurs paramètres : {grid_dt.best_params_}")

best_dt = grid_dt.best_estimator_
best_dt.fit(X_train_scaled, y_train)

y_pred_dt = best_dt.predict(X_test_scaled)

print(f"Rappel (Recall) : {recall_score(y_test, y_pred_dt):.4f}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}")
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred_dt))

 Meilleurs paramètres : {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
Rappel (Recall) : 0.8529
Accuracy : 0.8207
Matrice de confusion :
 [[64 18]
 [15 87]]


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, confusion_matrix, accuracy_score

rfc = RandomForestClassifier(class_weight='balanced', random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [4, 6, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(rfc, param_grid, cv=5, scoring='recall', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print("Meilleurs paramètres :", grid_search.best_params_)

best_rf = RandomForestClassifier(**grid_search.best_params_,
                                 class_weight='balanced',
                                 random_state=42)
best_rf.fit(X_train_scaled, y_train)

y_pred = best_rf.predict(X_test_scaled)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred))

Meilleurs paramètres : {'max_depth': 6, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 200}
Accuracy : 0.8152173913043478
Recall   : 0.8333333333333334
Matrice de confusion :
 [[65 17]
 [17 85]]


In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, confusion_matrix, accuracy_score

xgb = XGBClassifier(scale_pos_weight=1, random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_xgb = GridSearchCV(xgb, param_grid, cv=5, scoring='recall', n_jobs=-1)
grid_xgb.fit(X_train_scaled, y_train)

print("Meilleurs paramètres XGBoost :", grid_xgb.best_params_)

best_xgb = XGBClassifier(**grid_xgb.best_params_,
                         scale_pos_weight=1, random_state=42,
                         use_label_encoder=False, eval_metric='logloss')
best_xgb.fit(X_train_scaled, y_train)

y_pred = best_xgb.predict(X_test_scaled)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred))

Meilleurs paramètres XGBoost : {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Accuracy : 0.8152173913043478
Recall   : 0.8627450980392157
Matrice de confusion :
 [[62 20]
 [14 88]]


In [9]:
import joblib

joblib.dump(lr_final, '../models/lr_final.pkl')     
joblib.dump(best_dt, '../models/best_dt.pkl')  
joblib.dump(best_rf, '../models/best_rf.pkl')
joblib.dump(best_xgb, '../models/best_xgb.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']